# Stage 1 / Step 1 - Component 1: Content Quality Score

"Does this ad look like viral content?" -> a single score in [0, 1] from the post text only.

This is the lightweight stand-in for ViralBERT (TF-IDF + LogisticRegression on CPU; a fine-tuned
BERT is a later upgrade). It is trained on the rich YouTube dataset (`video_features.parquet`),
NOT on the minimal Kafka stream.

Output: an **out-of-fold** `content_score` per video, so it can feed the XGBoost fusion (Step 4)
without label leakage. We also save a model fitted on all data for serving.

In [4]:
# load features + build the binary viral label
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score
import joblib

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

# Historical notebook only: its retired relative top-25% label is not an official dataset-v3 target.
VIRAL_QUANTILE = 0.75
thr = df["virality_score"].quantile(VIRAL_QUANTILE)
df["is_viral"] = (df["virality_score"] >= thr).astype(int)

pos_rate = df["is_viral"].mean()
print(f"rows: {len(df)} | viral={df['is_viral'].sum()} | positive rate (PR-AUC baseline): {pos_rate:.3f}")

rows: 515 | viral=129 | positive rate (PR-AUC baseline): 0.250


In [5]:
# Component 1: text -> P(viral), computed OUT-OF-FOLD to avoid leakage downstream
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

content_clf = Pipeline([
    ("tf", TfidfVectorizer(max_features=2000, min_df=5, ngram_range=(1, 2),
                           stop_words="english", sublinear_tf=True)),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# out-of-fold probability of the positive (viral) class
oof_proba = cross_val_predict(content_clf, df["text_all"], df["is_viral"],
                              cv=cv, method="predict_proba")[:, 1]
df["content_score"] = oof_proba

ap = average_precision_score(df["is_viral"], oof_proba)
auc = roc_auc_score(df["is_viral"], oof_proba)
print(f"Component 1 content_score  PR-AUC: {ap:.3f}  (baseline {pos_rate:.3f})  |  ROC-AUC: {auc:.3f}")

Component 1 content_score  PR-AUC: 0.604  (baseline 0.250)  |  ROC-AUC: 0.830


In [6]:
# fit a final model on ALL data for serving, then save model + the OOF scores table
content_clf.fit(df["text_all"], df["is_viral"])

out_dir = ROOT / "ml" / "models"
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(content_clf, out_dir / "stage1_content_model.joblib")

stage1 = df[["video_id", "is_viral", "content_score"]].copy()
stage1.to_parquet(ROOT / "ml" / "data" / "stage1_scores.parquet", index=False)
print("Saved model ->", out_dir / "stage1_content_model.joblib")
print("Saved scores ->", ROOT / "ml" / "data" / "stage1_scores.parquet", "| shape:", stage1.shape)
stage1.head()

Saved model -> d:\USER-BEHAVIOR-SOCIAL-MEDIA\ml\models\stage1_content_model.joblib
Saved scores -> d:\USER-BEHAVIOR-SOCIAL-MEDIA\ml\data\stage1_scores.parquet | shape: (515, 3)


,video_id,is_viral,content_score
0,-1AmUBWRuLQ,0,0.613083
1,-3PKO--H8Do,0,0.349575
2,-7RCFxLojc8,0,0.426035
3,-EG6rqA2vvA,0,0.504464
4,-fD0OggBY4k,0,0.673051
